## But

Vérifier la validation du pipeline prévu dans AR1

## Principe général

* Historique complet **1959–2025** sur **UNRATE**.
* Données consommées exclusivement via **Feast**.
* Modèle Baseline et univarié.
* Backtesting temporel minimal avec prévisions ponctuelles et intervalles de prédiction conformes (95 %).
* Comparer AR1 et ARp

## Résultat
cf la fin

# Package

In [1]:
# ----------------------------
# Core
# ----------------------------
from pathlib import Path
import numpy as np
import pandas as pd

# ----------------------------
# Feature Store
# ----------------------------
from feast import FeatureStore

# ----------------------------
# Nixtla
# ----------------------------
from statsforecast import StatsForecast
from statsforecast.models import AutoRegressive
from statsforecast.utils import ConformalIntervals
from utilsforecast.plotting import plot_series

# Importation des données

In [2]:
# ----------------------------
# Locate Feast repo (notebook-safe)
# ----------------------------
def find_project_root(start: Path, marker: str = "2_data_processing") -> Path:
    p = start.resolve()
    for parent in [p] + list(p.parents):
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(
        f"Impossible de trouver la racine projet (marker '{marker}') depuis {start}"
    )

PROJECT_ROOT = find_project_root(Path.cwd(), marker="2_data_processing")

FEAST_REPO_PATH = (
    PROJECT_ROOT
    / "2_data_processing"
    / "feature_store"
    / "feast_repo"
    / "feature_repo"
)

print("FEAST_REPO_PATH:", FEAST_REPO_PATH)
print("feature_store.yaml exists:", (FEAST_REPO_PATH / "feature_store.yaml").exists())


# ----------------------------
# Load features from Feast
# ----------------------------
def load_features_from_feast(entity_df: pd.DataFrame, feature_refs: list[str]) -> pd.DataFrame:
    fs = FeatureStore(repo_path=str(FEAST_REPO_PATH))
    return fs.get_historical_features(entity_df=entity_df, features=feature_refs).to_df()


# ----------------------------
# Config data
# ----------------------------
START = "1959-01-01"
END   = "2025-09-01"
FREQ  = "MS"
SERIES_ID = "UNRATE"
FEATURE_REFS = ["stationary_value:value"]  # UNRATE déjà stationnaire

dates = pd.date_range(start=START, end=END, freq=FREQ)
entity_df = pd.DataFrame({"series_id": [SERIES_ID] * len(dates), "date": dates})

ts_raw = load_features_from_feast(entity_df, FEATURE_REFS)

ts = (
    ts_raw
    .rename(columns={"series_id": "unique_id", "date": "ds", "value": "y"})
    .sort_values(["unique_id", "ds"])
    .reset_index(drop=True)
)

FEAST_REPO_PATH: D:\Portofolio Data science\Time Series\Explainable_AI_Forecast_and_explain_the_Unemployment_of_USA\2_data_processing\feature_store\feast_repo\feature_repo
feature_store.yaml exists: True
Using date as the event timestamp. To specify a column explicitly, please name it event_timestamp.


# Dictionnaire de modèle

In [3]:
# ----------------------------
# (NEW) Dictionnaire modèle basé sur p*
# ----------------------------
SF_MODELS = {
    "AR_pstar": lambda: AutoRegressive(lags=p_star)
}

# Backtesting

In [16]:
from mlforecast.utils import PredictionIntervals

def run_backtesting_h12_simple(
    mlf,
    ts,
    *,
    h=12,
    step_size=12,
    partitions=4,
    pi_windows=3,
    levels=[95],
):
    """
    Simple backtesting:
    - horizon h (default: 12 months)
    - step_size between cutoffs (default: 12 months)
    - few partitions (default: 4)
    - conformal prediction intervals
    """

    pi = PredictionIntervals(
        h=h,
        n_windows=pi_windows,
        method="conformal_distribution",
    )

    bkt_df = mlf.cross_validation(
        df=ts,
        h=h,
        step_size=step_size,
        n_windows=partitions,
        prediction_intervals=pi,
        level=levels,
        fitted=True,
    )

    return bkt_df

# Run 

In [17]:

# ----------------------------
# Config
# ----------------------------
START = "1959-01-01"
END   = "2025-09-01"
FREQ  = "MS"
SERIES_ID = "UNRATE"

FEATURE_REFS = ["stationary_value:value"]  # y uniquement

H = 12
STEP_SIZE = 12
TEST_START = "1990-01-01"
PARTITIONS = 35
PI_WINDOWS = 3
LEVELS = [95]

AR_LAGS = 12 # Ajout

P_SELECTION_START = pd.Timestamp("1983-01-01", tz="UTC") # Ajout

p_grid = list(range(1, 13))  # ex: 1..12

In [19]:
# ----------------------------
# (NEW) Sélection de p* par MAE rolling maison (TRAIN only: avant TEST_START)
# ----------------------------
def mae_rolling_house(
    ts: pd.DataFrame,
    p: int,
    *,
    h: int,
    step_size: int,
    test_start_ts: pd.Timestamp,
    freq: str
) -> float:
    # 1) TRAIN ONLY (évite leakage)
    ts_train = ts[ts["ds"] < test_start_ts].copy()
    ts_train = ts_train.sort_values(["unique_id", "ds"]).reset_index(drop=True)

    ds_sorted = ts_train["ds"].sort_values().reset_index(drop=True)
    N = len(ds_sorted)

    # 2) cutoffs possibles:
    # - on commence la validation (MAE) à partir de 1983-01
    # - on doit pouvoir prévoir h pas et avoir la vérité sur ces h pas (dans train)
    last_cutoff_idx = N - 1 - h
    if last_cutoff_idx <= p:
        raise ValueError("Pas assez de points dans TRAIN pour évaluer ce p avec h donné.")

    first_cutoff_idx = ds_sorted[ds_sorted >= P_SELECTION_START].index.min()
    if pd.isna(first_cutoff_idx):
        raise ValueError("P_SELECTION_START est après la fin de la période TRAIN.")
    first_cutoff_idx = int(first_cutoff_idx)

    cutoff_indices = list(range(first_cutoff_idx, last_cutoff_idx + 1, step_size))
    if len(cutoff_indices) == 0:
        raise ValueError("Aucun cutoff généré (vérifie 1983-01, h, step_size, taille TRAIN).")

    maes = []
    for cut_idx in cutoff_indices:
        cutoff_date = ds_sorted.iloc[cut_idx]

        # expanding window: 1959 -> cutoff_date
        train_df = ts_train[ts_train["ds"] <= cutoff_date].copy()

        sf_tmp = StatsForecast(models=[AutoRegressive(lags=p)], freq=freq)
        fcst_df = sf_tmp.forecast(df=train_df, h=h)

        # colonne du modèle dans fcst_df (souvent "AutoRegressive")
        model_col = [c for c in fcst_df.columns if c.lower().startswith("autoregressive")][0]

        # vérité future (toujours dans TRAIN)
        y_true = ts_train[ts_train["ds"].isin(fcst_df["ds"])][["unique_id", "ds", "y"]]
        merged = fcst_df.merge(y_true, on=["unique_id", "ds"], how="inner")

        if len(merged) == 0:
            continue

        maes.append(float(np.mean(np.abs(merged["y"].values - merged[model_col].values))))

    if len(maes) == 0:
        raise ValueError("Impossible de calculer MAE (merge vide partout).")

    return float(np.mean(maes))


# ----------------------------
# Calcul de p*
# ----------------------------
test_start_ts = pd.Timestamp(TEST_START, tz="UTC")  # si TEST_START est une string

mae_by_p = {
    p: mae_rolling_house(ts, p, h=H, step_size=STEP_SIZE, test_start_ts=test_start_ts, freq=FREQ)
    for p in p_grid
}
p_star = min(mae_by_p, key=mae_by_p.get)

print("MAE_by_p =", mae_by_p)
print("p_star =", p_star, "| MAE =", mae_by_p[p_star])

# on écrase AR_LAGS pour le backtesting final
AR_LAGS = p_star

MAE_by_p = {1: 0.6481697837927555, 2: 0.6304693695213929, 3: 0.5845830753842959, 4: 0.5991043163898421, 5: 0.6044293687581733, 6: 0.5984936577028009, 7: 0.6082362345113641, 8: 0.6206989292054744, 9: 0.618831483855984, 10: 0.6119878200327701, 11: 0.6258197180602865, 12: 0.6200671431072822}
p_star = 3 | MAE = 0.5845830753842959


In [20]:
# ----------------------------
# 1) entity_df
# ----------------------------
dates = pd.date_range(start=START, end=END, freq=FREQ)
entity_df = pd.DataFrame({"series_id": [SERIES_ID] * len(dates), "date": dates})

In [9]:
# ----------------------------
# 2) Feast -> ts (format StatsForecast)
# ----------------------------
ts_raw = load_features_from_feast(entity_df=entity_df, feature_refs=FEATURE_REFS)

ts = (
    ts_raw
    .rename(columns={"series_id": "unique_id", "date": "ds", "value": "y"})
    .sort_values(["unique_id", "ds"])
    .reset_index(drop=True)
)

Using date as the event timestamp. To specify a column explicitly, please name it event_timestamp.


In [10]:
# ----------------------------
# 3) StatsForecast
# ----------------------------
sf = StatsForecast(
    models=[AutoRegressive(lags=AR_LAGS)],
    freq=FREQ,
)

In [11]:
# ----------------------------
# 4) Cross-validation + Conformal intervals
# ----------------------------
ci = ConformalIntervals(h=H, n_windows=PI_WINDOWS)

test_start_ts = pd.Timestamp(TEST_START, tz="UTC")
ds_sorted = ts["ds"].sort_values().reset_index(drop=True)

mask = ds_sorted < test_start_ts
if not mask.any():
    raise ValueError(f"TEST_START={TEST_START} est trop tôt (aucune date avant dans ts['ds']).")

cutoff_date = ds_sorted[mask].iloc[-1]
c = int(ds_sorted[ds_sorted == cutoff_date].index[0])
N = len(ds_sorted)

n_windows = int((((N - 1 - H) - c) // STEP_SIZE) + 1)
if n_windows <= 0:
    raise ValueError(f"TEST_START={TEST_START} est trop tard pour h={H} et step_size={STEP_SIZE}.")

bkt_df = sf.cross_validation(
    df=ts,
    h=H,
    step_size=STEP_SIZE,
    n_windows=n_windows,
    prediction_intervals=ci,
    level=LEVELS,
)

# Garder uniquement la partie test à partir de TEST_START
bkt_df = bkt_df[bkt_df["ds"] >= test_start_ts].reset_index(drop=True)

In [12]:
# ----------------------------
# 5) Reusable output table (FINAL)
# ----------------------------
# NB: le nom de la colonne modèle dépend du "alias" StatsForecast.
# Par défaut c'est souvent "AutoRegressive" (ou un nom proche).
model_col = [c for c in bkt_df.columns if c.lower().startswith("autoregressive")][0]

lo_col = [c for c in bkt_df.columns
          if c.lower().startswith("autoregressive") and c.lower().endswith("lo-95")][0]
hi_col = [c for c in bkt_df.columns
          if c.lower().startswith("autoregressive") and c.lower().endswith("hi-95")][0]

df_ar_forecasts = (
    bkt_df[["unique_id", "ds", "cutoff", "y", model_col, lo_col, hi_col]]
    .rename(columns={
        "unique_id": "series_id",
        "ds": "date",
        "y": "y_obs",
        model_col: "y_hat_ar",
        lo_col: "y_hat_ar_lo_95",
        hi_col: "y_hat_ar_hi_95",
    })
    # dates propres (sans +00:00), pratique pour plots / export
    .assign(
        date=lambda d: pd.to_datetime(d["date"]).dt.tz_localize(None),
        cutoff=lambda d: pd.to_datetime(d["cutoff"]).dt.tz_localize(None),
    )
    .sort_values(["series_id", "date"])
    .reset_index(drop=True)
)

df_ar_forecasts

,series_id,date,cutoff,y_obs,y_hat_ar,y_hat_ar_lo_95,y_hat_ar_hi_95
0,UNRATE,1990-10-01,1990-09-01,0.6,0.642696,0.436228,0.849164
1,UNRATE,1990-11-01,1990-09-01,0.8,0.644852,0.506317,0.783386
2,UNRATE,1990-12-01,1990-09-01,0.9,0.624565,0.477874,0.771255
3,UNRATE,1991-01-01,1990-09-01,1.0,0.589918,0.547542,0.632294
4,UNRATE,1991-02-01,1990-09-01,1.3,0.548483,0.326881,0.770084
...,...,...,...,...,...,...,...
415,UNRATE,2025-05-01,2024-09-01,0.2,0.095674,-0.846188,1.037535
416,UNRATE,2025-06-01,2024-09-01,0.0,0.081720,-1.059061,1.222500
417,UNRATE,2025-07-01,2024-09-01,0.0,0.069349,-0.849621,0.988318
418,UNRATE,2025-08-01,2024-09-01,0.1,0.058381,-0.580519,0.697281


# Graphique

In [13]:
df_obs = (
    df_ar_forecasts
    .rename(columns={
        "series_id": "unique_id",
        "date": "ds",
        "y_obs": "y",
    })
    [["unique_id", "ds", "y"]]
)

In [14]:
df_fcst = (
    df_ar_forecasts
    .rename(columns={
        "series_id": "unique_id",
        "date": "ds",
        "y_hat_ar": "AR",
        "y_hat_ar_lo_95": "AR-lo-95",
        "y_hat_ar_hi_95": "AR-hi-95",
    })
    [[
        "unique_id",
        "ds",
        "AR",
        "AR-lo-95",
        "AR-hi-95",
    ]]
)

In [15]:
from utilsforecast.plotting import plot_series

fig = plot_series(
    df=df_obs,
    forecasts_df=df_fcst,
    level=[95],
    engine="plotly",
).update_layout(height=400)

# Rename legend entries
for trace in fig.data:
    if trace.name == "y":
        trace.name = "Unemployment rate (%)"
    elif trace.name == "AR":
        trace.name = "AutoRegressive (AR)"
    elif "level_95" in trace.name.lower():
        trace.name = "95% Prediction Interval"

fig.show()

## Résultat
Globalement, le modèle auto-régressif reste proche des observations en période de stabilité. C'est une bonne capacité à capter la dynamique du chômage.

Lors des ruptures structurelles (crise de 2008, Covid-19), la qualité des prévisions se dégrade et les intervalles de prédiction s’élargissent. Ce qui qui traduit une incertitude de plus en plus élevée.

Le modèle capte la direction des variations, mais sa fiabilité diminue en période de crise, sans masquer cette incertitude.

Cette étude constitue un **sanity check du système de prévision**. Elle valide le comportement attendu du modèle et la cohérence du pipeline. Une approche plus complexe est attendu. 

## Next
Essayons d'optimiser le paramètre "p" de AR pour confirmer notre analyse. 